# Cardiovascular Disease Prediction
## Notebook 02: Random Forest Classifier
---
**Author:** MS26912448 (Member 1)
**Algorithm:** Random Forest (Ensemble – Bagging of Decision Trees)

### Algorithm Background

A **Random Forest** is an ensemble learning method that constructs multiple decision trees
during training and outputs the **mode** (majority vote) of their predictions.

#### Key Concepts:
- **Bootstrap Aggregation (Bagging):** Each tree is trained on a random sample (with replacement)
  of the training data, reducing variance.
- **Feature Randomness:** At each split, only a random subset of `√n` features is considered,
  decorrelating trees and reducing correlation between them.
- **Ensemble Prediction:** Final class = majority vote across all trees, reducing overfitting.

#### Why Random Forest for this Dataset?
1. Handles **mixed feature types** (binary, categorical, continuous) natively
2. **Robust to outliers** — uses splits, not distances
3. Provides **feature importance** — useful for clinical interpretation
4. Generally **does not require feature scaling**
5. Strong baseline performance on tabular medical data

In [ ]:
# Import all required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import os
import warnings

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, cross_val_score, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    classification_report, roc_curve, ConfusionMatrixDisplay
)

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
os.makedirs('plots', exist_ok=True)

print("✓ Libraries imported.")

In [ ]:
import os

# ============================================================
# ENVIRONMENT SETUP — comment out the block you are NOT using
# ============================================================

# ── OPTION A: Google Colab ───────────────────────────────────
# Mounts Drive and changes to your project folder so
# preprocessed_data.pkl (saved by NB01) can be found here.
from google.colab import drive
drive.mount('/content/drive')
DRIVE_PATH = '/content/drive/MyDrive/ML-Assignment/attemp2'
os.chdir(DRIVE_PATH)
print(f"✓ Working directory: {DRIVE_PATH}")

# ── OPTION B: Local Jupyter ──────────────────────────────────
# Comment out Option A above and uncomment the line below.
# All pkl files must be in the same folder as this notebook.
# LOCAL_PATH = os.path.dirname(os.path.abspath("__file__"))
# os.chdir(LOCAL_PATH)


In [ ]:
# Load preprocessed data saved by Notebook 01
with open('preprocessed_data.pkl', 'rb') as f:
    data = pickle.load(f)

X_train      = data['X_train']
X_test       = data['X_test']
y_train      = data['y_train']
y_test       = data['y_test']
feature_names = data['feature_names']

print(f"✓ Data loaded successfully.")
print(f"  Training samples : {X_train.shape[0]:,}  |  Features: {X_train.shape[1]}")
print(f"  Test samples     : {X_test.shape[0]:,}")
print(f"  Feature names    : {feature_names}")

## 1. Baseline Random Forest Model

We first train a baseline model with default hyperparameters to establish a starting performance.

In [ ]:
# Baseline Random Forest with default parameters
rf_baseline = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_baseline.fit(X_train, y_train)

y_pred_base = rf_baseline.predict(X_test)
y_prob_base = rf_baseline.predict_proba(X_test)[:, 1]

# Baseline metrics
acc  = accuracy_score(y_test, y_pred_base)
prec = precision_score(y_test, y_pred_base)
rec  = recall_score(y_test, y_pred_base)
f1   = f1_score(y_test, y_pred_base)
auc  = roc_auc_score(y_test, y_prob_base)

print("=== Baseline Random Forest Performance ===")
print(f"  Accuracy  : {acc:.4f}  ({acc*100:.2f}%)")
print(f"  Precision : {prec:.4f}")
print(f"  Recall    : {rec:.4f}")
print(f"  F1-Score  : {f1:.4f}")
print(f"  ROC-AUC   : {auc:.4f}")
print("\n", classification_report(y_test, y_pred_base, target_names=['No Disease','Has Disease']))

## 2. Hyperparameter Tuning with GridSearchCV

We tune key hyperparameters using **5-Fold Stratified Cross-Validation** optimising for **F1-score**
(better than accuracy for imbalanced classes).

| Parameter         | Values Tested         | Description                              |
|------------------|-----------------------|------------------------------------------|
| `n_estimators`   | 100, 200, 300         | Number of trees in the forest            |
| `max_depth`      | 10, 20, None          | Maximum depth of each tree               |
| `min_samples_split` | 2, 5              | Minimum samples to split a node          |
| `max_features`   | 'sqrt', 'log2'        | Features considered at each split        |

In [ ]:
# Hyperparameter grid — kept compact for faster tuning
# (3-fold CV × 12 candidates = 36 fits, runs in ~1-2 min)
param_grid = {
    'n_estimators':      [100, 200],
    'max_depth':         [10, 20, None],
    'min_samples_split': [2, 5],
    'max_features':      ['sqrt']
}

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

print("Starting GridSearchCV (3-fold, 12 candidates = 36 fits)...")
grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    param_grid,
    cv=cv,
    scoring='f1',
    n_jobs=-1,
    verbose=1
)
grid_search.fit(X_train, y_train)

print(f"\n✓ GridSearchCV complete.")
print(f"  Best parameters : {grid_search.best_params_}")
print(f"  Best CV F1      : {grid_search.best_score_:.4f}")


In [ ]:
# Retrieve and evaluate the best model
best_rf = grid_search.best_estimator_

print("=== Best RF Parameters ===")
for param, val in grid_search.best_params_.items():
    print(f"  {param}: {val}")

## 3. Final Model Evaluation

### 3.1 Performance Metrics

In [ ]:
# Evaluate best model on test set
y_pred_rf = best_rf.predict(X_test)
y_prob_rf  = best_rf.predict_proba(X_test)[:, 1]

# Compute all metrics
rf_metrics = {
    'accuracy':  accuracy_score(y_test, y_pred_rf),
    'precision': precision_score(y_test, y_pred_rf),
    'recall':    recall_score(y_test, y_pred_rf),
    'f1':        f1_score(y_test, y_pred_rf),
    'roc_auc':   roc_auc_score(y_test, y_prob_rf)
}

print("=== Random Forest – Final Test Results ===")
for metric, val in rf_metrics.items():
    bar = '█' * int(val * 30)
    print(f"  {metric:<12}: {val:.4f}  {bar}")

print("\n=== Detailed Classification Report ===")
print(classification_report(y_test, y_pred_rf, target_names=['No Disease', 'Has Disease']))